# Task 2C — Base vs fine-tuned evaluation, plus the RAG fallback bonus (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazifmhd/Task-1---Financial-AI/blob/main/task2_genai/03_evaluate.ipynb)

**Fairness:** both models get the **same system prompt**, the **same held-out test set** (never seen in
training), the same 4-bit NF4 loading, and greedy decoding with the same stop tokens. The only variable is fine-tuning.
The fine-tuned model is loaded **from the public Hugging Face repo**, which also proves the published link works.

In [15]:
# Colab: clone the repo (data + src) and install pinned dependencies.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Task-1---Financial-AI"):
        subprocess.run(["git", "clone", "-q", "https://github.com/nazifmhd/Task-1---Financial-AI.git"], check=True)
    os.chdir("Task-1---Financial-AI/task2_genai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) != "task2_genai" and os.path.isdir("task2_genai"):
    os.chdir("task2_genai")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| Colab:", IN_COLAB)

cwd: /content/Task-1---Financial-AI/task2_genai/Task-1---Financial-AI/task2_genai | Colab: True


In [16]:
import json, logging, warnings, time
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
for n in ("httpx", "urllib3", "sentence_transformers", "chromadb", "huggingface_hub", "datasets"):
    logging.getLogger(n).setLevel(logging.WARNING)
import torch, transformers, peft, bitsandbytes, pandas as pd
from IPython.display import display, Markdown, Image
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
import config
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GiB | compute capability {p.major}.{p.minor}")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)

GPU: Tesla T4 | 14.6 GiB | compute capability 7.5
torch 2.11.0+cu130 | transformers 4.53.2 | peft 0.16.0 | bitsandbytes 0.50.2


In [17]:
# HF token from Colab Secrets (key icon) - never printed, never committed.
from src.llm import read_secret
HF_TOKEN = read_secret("HF_TOKEN")
print("HF token loaded:", bool(HF_TOKEN), "| target repo:", config.HF_REPO_ID)

HF token loaded: True | target repo: nazifmhd/phi3-mini-compliance-extractor


In [18]:
from transformers import AutoModelForCausalLM
from src import train as T, evaluate as E
from src.format_split import load_split
test, val, train_rows = load_split("test"), load_split("val"), load_split("train")
print({"train": len(train_rows), "val": len(val), "test": len(test)})

def load_eval_model(name):
    m = AutoModelForCausalLM.from_pretrained(name, quantization_config=T.bnb_config(), device_map={"": 0},
                                             torch_dtype=torch.float16, attn_implementation="sdpa")
    m.eval(); return m, T.load_tokenizer(name)

{'train': 192, 'val': 24, 'test': 24}


## 1. Baseline: untuned Phi-3-mini with the same system prompt

In [19]:
print("System prompt used for BOTH models:\n" + E.STUDENT_SYSTEM_PROMPT + "\n")
base, tok_b = load_eval_model(config.BASE_MODEL)
t0 = time.time(); preds_base = E.predict_rows(base, tok_b, test); print(f"base: {time.time()-t0:.0f}s")
del base; T.free_gpu()

System prompt used for BOTH models:
You are a financial compliance extraction model. Read the clause and return ONLY a JSON object with exactly these keys in this order:
{"clause_type": one of [indemnification, termination, payment_terms, confidentiality, liability_cap, force_majeure, compliance_covenant, data_protection, aml_kyc, regulatory_reporting],
 "obligation": the primary duty the clause imposes, a concise paraphrase (max 40 words) using only facts in the clause,
 "party_responsible": the party that must perform it, copied exactly as named in the clause,
 "trigger_condition": the event that activates the obligation, or null if unconditional,
 "risk_flag": one of [low, medium, high]}
Never invent obligations, parties, amounts or deadlines that are not in the clause.



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

base: 32s


## 2. Fine-tuned (merged) model from the Hub

In [20]:
tuned, tok_t = load_eval_model(config.HF_REPO_ID)
t0 = time.time(); preds_tuned = E.predict_rows(tuned, tok_t, test); print(f"tuned: {time.time()-t0:.0f}s")
preds_val_tuned = E.predict_rows(tuned, tok_t, val)          # used only to calibrate the RAG threshold

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

tuned: 39s


## 3. ROUGE-L, BERTScore and structured metrics

In [21]:
sb, st = E.score_predictions(test, preds_base), E.score_predictions(test, preds_tuned)
labels = {"rougeL": "ROUGE-L (F)", "bertscore_f1": "BERTScore F1", "json_parse": "JSON parse rate",
          "schema_valid": "Schema-valid rate", "clause_type_acc": "clause_type accuracy",
          "risk_flag_acc": "risk_flag accuracy", "party_match": "party exact match",
          "trigger_null_agree": "trigger null/non-null agreement", "party_in_text": "party grounded in clause",
          "obligation_rougeL": "obligation-field ROUGE-L"}
rows_ = []
for k, name in labels.items():
    a = [float(d[k]) for d in sb["per_example"]]; b = [float(d[k]) for d in st["per_example"]]
    mean, lo, hi = E.bootstrap_diff_ci(a, b)
    rows_.append({"metric": name, "base": sb["aggregate"][k], "fine-tuned": st["aggregate"][k],
                  "Δ": mean, "95% CI of Δ (paired bootstrap)": f"[{lo:+.3f}, {hi:+.3f}]"})
results = pd.DataFrame(rows_)
display(Markdown(f"**Held-out test set: n = {len(test)}**"))
display(results.style.format({"base": "{:.3f}", "fine-tuned": "{:.3f}", "Δ": "{:+.3f}"}).hide(axis="index"))

INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO absl: Using default tokenizer.
INFO

**Held-out test set: n = 24**

metric,base,fine-tuned,Δ,95% CI of Δ (paired bootstrap)
ROUGE-L (F),0.633,0.778,+0.145,"[+0.094, +0.195]"
BERTScore F1,0.905,0.967,+0.062,"[+0.054, +0.069]"
JSON parse rate,0.792,1.000,+0.208,"[+0.083, +0.375]"
Schema-valid rate,0.750,1.000,+0.250,"[+0.083, +0.417]"
clause_type accuracy,0.625,0.917,+0.292,"[+0.125, +0.458]"
risk_flag accuracy,0.458,0.625,+0.167,"[-0.042, +0.375]"
party exact match,0.125,0.958,+0.833,"[+0.667, +0.958]"
trigger null/non-null agreement,0.500,0.708,+0.208,"[+0.083, +0.375]"
party grounded in clause,0.750,1.000,+0.250,"[+0.083, +0.417]"
obligation-field ROUGE-L,0.356,0.721,+0.365,"[+0.259, +0.473]"


In [22]:
print("| Model | ROUGE-L | BERTScore F1 |\n|---|---|---|")
print(f"| Phi-3-mini base (system prompt, no fine-tuning) | {sb['aggregate']['rougeL']:.3f} | {sb['aggregate']['bertscore_f1']:.3f} |")
print(f"| Phi-3-mini QLoRA fine-tuned | {st['aggregate']['rougeL']:.3f} | {st['aggregate']['bertscore_f1']:.3f} |")

| Model | ROUGE-L | BERTScore F1 |
|---|---|---|
| Phi-3-mini base (system prompt, no fine-tuning) | 0.633 | 0.905 |
| Phi-3-mini QLoRA fine-tuned | 0.778 | 0.967 |


## 4. Side-by-side outputs

In [23]:
side = pd.DataFrame({"id": [r["id"] for r in test],
                     "clause": [r["messages"][1]["content"][:150] for r in test],
                     "base": [p["raw"][:220] for p in preds_base],
                     "fine-tuned": [p["raw"][:220] for p in preds_tuned],
                     "reference": [r["messages"][2]["content"][:220] for r in test]})
display(side.head(8))

,id,clause,base,fine-tuned,reference
0,ex052,"If a covered loss occurs, the Insured shall pay to the Insurer a deductible of US$5,000 within five Business Days of receiving the claim...","```json\n{\n ""clause_type"": ""payment_terms"",\n ""obligation"": ""The Insured must pay a deductible of US$5,000 to the Insurer upon a cove...","{""clause_type"": ""payment_terms"", ""obligation"": ""pay a deductible of US$5,000 within five Business Days of receiving the claim notice"", ""...","{""clause_type"": ""payment_terms"", ""obligation"": ""pay a deductible of US$5,000"", ""party_responsible"": ""the Insured"", ""trigger_condition"": ..."
1,ex072,"The Borrower shall, at all times during the Term, implement and maintain appropriate technical and organisational measures to protect Pe...","```json\n{\n ""clause_type"": ""data_protection"",\n ""obligation"": ""Borrower must maintain data protection measures and notify Lender of b...","{""clause_type"": ""data_protection"", ""obligation"": ""implement and maintain appropriate technical and organisational measures to protect Pe...","{""clause_type"": ""data_protection"", ""obligation"": ""implement and maintain appropriate technical and organisational measures to protect Pe..."
2,ex214,"The Borrower covenants that, if any Borrower‑initiated transaction under the Syndicated Loan Facility exceeds USD 5,000,000, it shall im...","```json\n{\n ""clause_type"": ""aml_kyc"",\n ""obligation"": ""Borrower must implement enhanced AML procedures, verify counterparty, and subm...","{""clause_type"": ""aml_kyc"", ""obligation"": ""implement enhanced AML procedures, obtain independent verification of the counter‑party, and s...","{""clause_type"": ""compliance_covenant"", ""obligation"": ""implement enhanced AML procedures, obtain independent verification and submit a co..."
3,ex165,"THIS LOAN AGREEMENT may be terminated by the Lender Parties upon the occurrence of an Event of Default as defined in Clause 5.1, PROVIDE...","```json\n{\n ""clause_type"": ""termination"",\n ""obligation"": ""Lender Parties may terminate the loan if a default occurs, except for amou...","{""clause_type"": ""termination"", ""obligation"": ""terminate the loan agreement upon an Event of Default"", ""party_responsible"": ""the Lender P...","{""clause_type"": ""termination"", ""obligation"": ""Terminate the loan agreement upon an Event of Default, except for amounts already disburse..."
4,ex111,Confidentiality. The Insured shall keep strictly confidential all Confidential Information (as defined in Clause 5.2) received from the ...,"{""clause_type"": ""confidentiality"",\n ""obligation"": ""Insured must keep all confidential information strictly confidential and not disclos...","{""clause_type"": ""confidentiality"", ""obligation"": ""keep Confidential Information strictly confidential and not disclose to third parties ...","{""clause_type"": ""confidentiality"", ""obligation"": ""keep strictly confidential all Confidential Information received and not disclose to a..."
5,ex000,"The Borrower shall, upon any material change in its regulatory capital adequacy ratio, promptly submit to the Lender a certified complia...","```json\n{\n ""clause_type"": ""compliance_covenant"",\n ""obligation"": ""Submit a certified compliance report to the Lender proving continu...","{""clause_type"": ""compliance_covenant"", ""obligation"": ""submit a certified compliance report demonstrating continued minimum 8% Tier 1 cap...","{""clause_type"": ""compliance_covenant"", ""obligation"": ""submit a certified compliance report demonstrating continued meeting of the minimu..."
6,ex114,"The Counterparty shall, within thirty (30) days after the close of each calendar quarter, submit to the Central Bank of Sri Lanka a deta...","```json\n{\n ""clause_type"": ""compliance_covenant"",\n ""obligation"": ""Submit a detailed report of derivative transactions to the Central...","{""clause_type"": ""regulatory_reporting"", ""obligation"": ""submit a detailed re

## 5. Bonus — RAG fallback for low-confidence answers
* **Confidence** is the mean log-probability of the generated tokens *given the prompt* (from generation scores).
* **Threshold** is the 20th percentile of that confidence on the **validation** set, so roughly the least-confident fifth triggers retrieval. Any schema-invalid answer also triggers it.
* **Knowledge base:** ChromaDB with MiniLM embeddings over the **training** clauses and their gold extractions, plus label definitions and the risk rubric. There is no val/test leakage.

In [24]:
from src.rag_fallback import ComplianceKB, calibrate_threshold, answer_with_fallback
kb = ComplianceKB(train_rows)
thr = calibrate_threshold([p["mean_logprob"] for p in preds_val_tuned])
print(f"KB documents: {kb.col.count()} | confidence threshold (val p{config.RAG_CONFIDENCE_PERCENTILE}): {thr:.4f}")
rag = [answer_with_fallback(tuned, tok_t, r["messages"][1]["content"], kb, thr, first=p)
       for r, p in zip(test, preds_tuned)]
preds_rag = [{"id": r["id"], **x["final"]} for r, x in zip(test, rag)]
s_rag = E.score_predictions(test, preds_rag, with_bertscore=False)
n_fb = sum(x["used_rag"] for x in rag)
print(f"Fallback triggered on {n_fb}/{len(test)} test clauses")
cmp = pd.DataFrame([{"pipeline": "fine-tuned", **{labels[k]: st["aggregate"][k] for k in ("rougeL", "schema_valid", "clause_type_acc", "risk_flag_acc", "party_match")}},
                    {"pipeline": "fine-tuned + RAG fallback", **{labels[k]: s_rag["aggregate"][k] for k in ("rougeL", "schema_valid", "clause_type_acc", "risk_flag_acc", "party_match")}}])
display(cmp.round(3))

KB documents: 192 | confidence threshold (val p20): -0.0891


INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.
INFO absl: Using default tokenizer.


Fallback triggered on 7/24 test clauses


,pipeline,ROUGE-L (F),Schema-valid rate,clause_type accuracy,risk_flag accuracy,party exact match
0,fine-tuned,0.778,1.0,0.917,0.625,0.958
1,fine-tuned + RAG fallback,0.775,1.0,0.917,0.542,0.958


In [25]:
# Concrete before/after: the fallback case with the largest gain (ROUGE-L + label correctness).
per_t = {d["id"]: d for d in st["per_example"]}; per_r = {d["id"]: d for d in s_rag["per_example"]}
def gain(i): return (per_r[i]["rougeL"] + per_r[i]["clause_type_acc"] + per_r[i]["risk_flag_acc"]) - \
                    (per_t[i]["rougeL"] + per_t[i]["clause_type_acc"] + per_t[i]["risk_flag_acc"])
fb = [(r, x) for r, x in zip(test, rag) if x["used_rag"]]
if fb:
    r, x = max(fb, key=lambda rx: gain(rx[0]["id"]))
    print(f"[{r['id']}] confidence {x['first']['mean_logprob']:.4f} < threshold {thr:.4f} -> retrieved {x['retrieved']}")
    print("\nCLAUSE:\n" + r["messages"][1]["content"])
    print("\nBEFORE (fine-tuned only):\n" + x["first"]["raw"])
    print("\nAFTER (with retrieved context):\n" + x["final"]["raw"])
    print("\nREFERENCE:\n" + r["messages"][2]["content"])
    print(f"\nROUGE-L {per_t[r['id']]['rougeL']:.3f} -> {per_r[r['id']]['rougeL']:.3f} | "
          f"clause_type {per_t[r['id']]['clause_type_acc']:.0f} -> {per_r[r['id']]['clause_type_acc']:.0f} | "
          f"risk {per_t[r['id']]['risk_flag_acc']:.0f} -> {per_r[r['id']]['risk_flag_acc']:.0f}")

[ex098] confidence -0.1065 < threshold -0.0891 -> retrieved [{'id': 'ex021', 'clause_type': 'force_majeure', 'distance': 0.16048991680145264}, {'id': 'ex161', 'clause_type': 'force_majeure', 'distance': 0.2113248109817505}, {'id': 'ex118', 'clause_type': 'force_majeure', 'distance': 0.2951822280883789}]

CLAUSE:
Neither Party shall be liable for any failure or delay in performing its obligations under this Agreement to the extent such failure or delay is caused by events beyond its reasonable control, including but not limited to acts of God, war, terrorism, civil unrest, governmental action, or any other force majeure event. Each Party shall promptly notify the other of the occurrence of any such event.

BEFORE (fine-tuned only):
{"clause_type": "force_majeure", "obligation": "limit liability for failure or delay caused by uncontrollable events", "party_responsible": "Neither Party", "trigger_condition": "occurrence of any such event", "risk_flag": "high"}

AFTER (with retrieved conte

## 6. Save artifacts (used by notebook 04) and publish them next to the model

In [26]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
def dump(name, rows):
    with open(config.OUTPUT_DIR / name, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
dump("predictions_base.jsonl", preds_base); dump("predictions_tuned.jsonl", preds_tuned)
dump("rag_results.jsonl", [{"id": r["id"], **x} for r, x in zip(test, rag)])
json.dump({"n_test": len(test), "base": sb, "tuned": st, "tuned_rag": s_rag, "rag_threshold": thr,
           "rag_triggered": n_fb}, open(config.OUTPUT_DIR / "metrics.json", "w"), indent=1)
results.to_csv(config.OUTPUT_DIR / "metrics_table.csv", index=False)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in ("predictions_base.jsonl", "predictions_tuned.jsonl", "rag_results.jsonl", "metrics.json", "metrics_table.csv"):
        api.upload_file(path_or_fileobj=str(config.OUTPUT_DIR / f), path_in_repo=f"eval/{f}", repo_id=config.HF_REPO_ID)
    print("Uploaded eval artifacts to", f"https://huggingface.co/{config.HF_REPO_ID}/tree/main/eval")

Uploaded eval artifacts to https://huggingface.co/nazifmhd/phi3-mini-compliance-extractor/tree/main/eval
